# Laya V6 — Native packed INT4 GPU inference (not FP16 reconstruction)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V6_Native_INT4_Fast_GPU_Colab.ipynb)

**Goal:** real GPU packed INT4 GEMM with TorchAO and optional GemLite, measured on Laya's ModernBERT encoder. Qualify candidates only if native operator latency beats FP16 at **M=512 and 2048**, and cumulative decision quality passes disjoint development selection/guard sets. Sealed audit and end-to-end benchmarking run after selecting modules.

**Hardware:** prefer **L4 / A100 / H100**. Tesla T4 might not support the same fast kernels. TinyGEMM can be faster at M=1 but slower at larger matrix batches: do not extrapolate LLM token-decode speed to encoder prefill. If no faster native INT4 kernel exists, retain FP16 and report that clearly.

**Source checkpoint:** Defaults to the original Laya 28-layer standalone FP16 model. To start with your V5 weights, set `STARTING_REPO` to the **already uploaded V5 standalone FP16 model**, not the delta-only package. Native TorchAO INT4 **does not execute V5's earlier sparse-packed format directly**; requantizing changes weights again.

This is a benchmark/research Colab. It does not publish a model before a compatible TorchAO loader and independent reproducibility test exist.


In [ ]:
#@title 1. Install native INT4 support (TorchAO)
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4" "datasets>=2.20,<5" "torchao>=0.15,<0.18" pandas
# Optional GemLite: %pip -q install gemlite


In [ ]:
#@title 2. Hardware, model and benchmark settings
import sys,os,json,random,copy,math,gc,zipfile,importlib.util
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from huggingface_hub import HfApi,snapshot_download
from urllib.request import urlopen
from IPython.display import display
STARTING_REPO="vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
GROUP_SIZE=128 #@param [32,64,128,256] {type:"raw"}
OPTIONAL_GEMLITE=True #@param {type:"boolean"}
MIN_OPERATOR_GAIN=1.10 #@param {type:"number"}
MIN_MODEL_GAIN=1.05 #@param {type:"number"}
MAX_TRIALS=64 #@param {type:"integer"}
MAX_KEEP=32 #@param {type:"integer"}
TRAIN_CASES=1200 #@param {type:"integer"}
DEV_COUNT=256 #@param {type:"integer"}
GUARD_COUNT=128 #@param {type:"integer"}
AUDIT_COUNT=384 #@param {type:"integer"}
MAX_DEV_ERRORS=2 #@param {type:"integer"}
MAX_GUARD_ERRORS=1 #@param {type:"integer"}
MAX_JS=0.0015 #@param {type:"number"}
BENCH_REPEATS=12 #@param {type:"integer"}
MAX_LEN=512
BATCH_SIZE=4
SEED=2061
OUT=Path("/content/Laya_Native_INT4_V6")
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Select L4/A100/H100, or test T4 (may be unsupported)"
GPU=torch.cuda.get_device_name(0)
SM=torch.cuda.get_device_capability(0)
if SM[0]<8:print("WARNING: pre-Ampere GPU; verify hardware INT4 compatibility")
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:HF_TOKEN=os.environ.get("HF_TOKEN")
SOURCE_SHA=HfApi(token=HF_TOKEN).model_info(STARTING_REPO).sha
ROOT=Path(snapshot_download(repo_id=STARTING_REPO,revision=SOURCE_SHA,token=HF_TOKEN))
meta=json.loads((ROOT/"config.json").read_text())
assert meta.get("format")=="laya-1p2-full-standalone-v1"
assert meta.get("dtype_on_disk")=="float16"
print("GPU:",GPU,"SM:",SM,"base:",STARTING_REPO,SOURCE_SHA)


In [ ]:
#@title 3. Exact standalone model + pinned native INT4 helper
def load_code(path,name):
    spec=importlib.util.spec_from_file_location(name,str(path))
    assert spec and spec.loader
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module;spec.loader.exec_module(module)
    return module
source=load_code(ROOT/"modeling_laya_standalone.py","laya_v6_source")
teacher,tokenizer=source.load_model(ROOT,device="cpu",local_files_only=True,compile_head=False)
teacher=teacher.to("cuda",dtype=torch.float16).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
try:
    if hasattr(teacher.encoder,"set_attn_implementation"):
        teacher.encoder.set_attn_implementation("sdpa")
    else:teacher.encoder.config._attn_implementation="sdpa"
except Exception as exc:print("SDPA preference:",repr(exc)[:140])
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
assert len(teacher.encoder.layers)==28
CODE_SHA="b5f791776573add5d47a2a1193b9e7354ebb1837"
code=Path("/content/laya_native_int4_v6.py")
with urlopen(f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{CODE_SHA}/src/tinycenn_lm/laya_native_int4_v6.py",timeout=90) as response:
    code.write_bytes(response.read())
assert code.stat().st_size>4000
native=load_code(code,"laya_native_int4_v6")
import torchao
print("TorchAO:",getattr(torchao,"__version__","unknown"))
# GemLite can be beneficial for larger encoder GEMMs. Install optionally.
# Installing is nonfatal: real kernel compatibility is checked in Cell 4.
if OPTIONAL_GEMLITE:
    try:
        import gemlite
    except ImportError:
        import subprocess
        print("Trying optional GemLite for batched native INT4...")
        installed=subprocess.run([sys.executable,"-m","pip","install","-q","gemlite"],
             capture_output=True,text=True,check=False)
        if installed.returncode:
            print("GemLite unavailable; skip:",installed.stderr[-350:])
CONFIGS=native.backend_configs(GROUP_SIZE,OPTIONAL_GEMLITE)
print("Candidate native packed INT4 backends:",list(CONFIGS))


In [ ]:
#@title 4. Verify actual packed INT4 kernels and rank them against FP16 on M=512/2048
TESTS=[]
SHAPES=("mlp.Wi","mlp.Wo","attn.Wqkv","attn.Wo")
for suffix in SHAPES:
    layername="encoder.layers.0."+suffix
    original=teacher.get_submodule(layername)
    for backend,cfg in CONFIGS.items():
        trial={"projection":suffix,"backend":backend,"compatible":False}
        try:
            candidate=native.quantize_linear(original,cfg)
            for xshape in ((512,original.in_features),
                           (4,512,original.in_features)):
                x=torch.randn(*xshape,dtype=torch.float16,device="cuda")
                with torch.inference_mode():y=candidate(x)
                assert y.shape==(*xshape[:-1],original.out_features)
                assert bool(torch.isfinite(y).all())
            tests=native.compare_linear_kernels(original,candidate,
                    m_values=(128,512,2048),repeats=BENCH_REPEATS)
            trial["compatible"]=True
            trial["latencies"]=tests
            trial["qualified"]=native.speed_gate(tests,min_gain=MIN_OPERATOR_GAIN,
                                                 target_m=(512,2048))
            trial["mean_gain"]=sum(row["speedup"] for row in tests
                                   if row["m"] in (512,2048))/2
            print(suffix,backend,"speedups",
                  [round(row["speedup"],2) for row in tests],
                  "QUALIFIED",trial["qualified"])
        except Exception as exc:
            trial["error"]=repr(exc)[:350]
            print("UNSUPPORTED",suffix,backend,repr(exc)[:140])
        TESTS.append(trial)
        if "candidate" in locals():del candidate
        gc.collect();torch.cuda.empty_cache()
(OUT/"operator_trials.json").write_text(json.dumps(TESTS,indent=2))
WINNERS={}
for trial in TESTS:
    if trial.get("qualified"):
        key=trial["projection"]
        if key not in WINNERS or WINNERS[key]["mean_gain"]<trial["mean_gain"]:
            WINNERS[key]=trial
print("Qualified INT4 encoder projection shape classes:",list(WINNERS))
if not WINNERS:print("NO qualified packed INT4 kernel on this GPU/workload. "
                     "A real no-speedup result is preferable to fake INT4.")


In [ ]:
#@title 5. Disjoint development/guard/audit data, without official-test tuning
from datasets import load_dataset
def to_obj(v):
    if isinstance(v,str):
        try:return json.loads(v)
        except Exception:return v
    return v
def extract(rows,n):
    per_case=[]
    for row in rows:
        state=to_obj(row.get("state"))
        questions=to_obj(row.get("questions"))
        if not isinstance(questions,dict):continue
        items=[]
        for question in questions.values():
            try:
                q=source.normalize_question(to_obj(question))
                ids,markers=source.build_sequence(tokenizer,state,q,max_len=MAX_LEN,
                    head_max_len=192,truncate_left=isinstance(state,list))
                if len(markers)<2:continue
                items.append(dict(ids=ids,markers=markers,
                                  qtype=source.QTYPES[q["t"]]))
            except (TypeError,ValueError,KeyError,IndexError):continue
        if items:per_case.append(items)
    results=[]
    for depth in range(max(map(len,per_case),default=0)):
        for case in per_case:
            if depth<len(case):
                results.append(case[depth])
                if len(results)>=n:return results
    return results
def to_batches(items,size=BATCH_SIZE,length=None):
    batches=[]
    for i in range(0,len(items),size):
        b=source.collate_items(items[i:i+size],int(tokenizer.pad_token_id))
        if length:
            pad=length-b["input_ids"].shape[1]
            if pad<0:raise ValueError("Example too long for benchmark")
            b["input_ids"]=F.pad(b["input_ids"],(0,pad),value=int(tokenizer.pad_token_id))
            b["attention_mask"]=F.pad(b["attention_mask"],(0,pad))
        batches.append({k:v.to("cuda") for k,v in b.items()})
    return batches
dataset=list(load_dataset("LocalLLaMA/typed-decisions","all",
                          split=f"train[:{TRAIN_CASES}]"))
random.Random(SEED).shuffle(dataset)
cut=[0,int(len(dataset)*.10),int(len(dataset)*.40),
     int(len(dataset)*.55),len(dataset)]
SELECT_ITEMS=extract(dataset[cut[1]:cut[2]],DEV_COUNT)
GUARD_ITEMS=extract(dataset[cut[2]:cut[3]],GUARD_COUNT)
AUDIT_ITEMS=extract(dataset[cut[3]:],AUDIT_COUNT)
assert [len(SELECT_ITEMS),len(GUARD_ITEMS),len(AUDIT_ITEMS)]==[
    DEV_COUNT,GUARD_COUNT,AUDIT_COUNT], "Adjust counts for dataset source-case availability"
SELECT,GUARD,AUDIT=[to_batches(x) for x in
    (SELECT_ITEMS,GUARD_ITEMS,AUDIT_ITEMS)]
@torch.inference_mode()
def logits(model,b):
    with torch.autocast("cuda",dtype=torch.float16):
        result=model(**b)
    return result[0] if isinstance(result,(tuple,list)) else result
@torch.inference_mode()
def references(batches):
    return [logits(teacher,b).float().cpu() for b in batches]
REF_S,REF_G,REF_A=[references(x) for x in (SELECT,GUARD,AUDIT)]
@torch.inference_mode()
def quality(model,batches,ref):
    errors=n=0;js_sum=0.
    for b,original in zip(batches,ref):
        output=logits(model,b).float().cpu()
        valid=b["marker_mask"].bool().cpu()
        a=output.masked_fill(~valid,-1e4)
        t=original.masked_fill(~valid,-1e4)
        p=a.softmax(-1).clamp_min(1e-9)
        q=t.softmax(-1).clamp_min(1e-9)
        mid=(p+q)*0.5
        js=0.5*(p*(p/mid).log()).sum(-1)+0.5*(q*(q/mid).log()).sum(-1)
        errors+=int((p.argmax(-1)!=q.argmax(-1)).sum())
        n+=len(p);js_sum+=float(js.sum())
    return dict(n=n,agreement=1-errors/n,errors=errors,mean_js=js_sum/n)
print("Teacher self-agreement:",quality(teacher,SELECT,REF_S))


In [ ]:
#@title 6. Accept only modules with REAL INT4 speed and model quality
student=copy.deepcopy(teacher).eval().requires_grad_(False)
accepted=[];trials=[]
for name in native.candidate_names(teacher)[:MAX_TRIALS]:
    if len(accepted)>=MAX_KEEP:break
    suffix=name.split(".",3)[-1]
    if suffix not in WINNERS:continue
    backend=WINNERS[suffix]["backend"]
    old=student.get_submodule(name)
    row={"module":name,"backend":backend,"accepted":False}
    try:
        packed=native.quantize_linear(old,CONFIGS[backend])
        native.swap_module(student,name,packed)
        dev=quality(student,SELECT,REF_S)
        guard=quality(student,GUARD,REF_G)
        good=(dev["errors"]<=MAX_DEV_ERRORS and
              guard["errors"]<=MAX_GUARD_ERRORS and
              dev["mean_js"]<=MAX_JS and guard["mean_js"]<=MAX_JS)
        row.update(selection=dev,guard=guard,accepted=bool(good))
        if good:
            accepted.append(dict(name=name,backend=backend,
                                 op_mean_gain=WINNERS[suffix]["mean_gain"]))
            print("ACCEPT native INT4:",name,dev["agreement"],guard["agreement"])
        else:
            native.swap_module(student,name,old)
    except Exception as exc:
        native.swap_module(student,name,old)
        row["error"]=repr(exc)[:260]
        print("Rejected",name,row["error"][:140])
    trials.append(row)
    if len(trials)%4==0:
        (OUT/"module_trials.json").write_text(json.dumps(trials,indent=2))
(OUT/"module_trials.json").write_text(json.dumps(trials,indent=2))
print("Actual packed INT4 modules accepted:",len(accepted))


In [ ]:
#@title 7. Sealed audit, end-to-end speed and deployability gates
FINAL_DEV=quality(student,SELECT,REF_S)
FINAL_GUARD=quality(student,GUARD,REF_G)
FINAL_AUDIT=quality(student,AUDIT,REF_A)
AUDIT_PASS=(FINAL_AUDIT["agreement"]>=.99 and FINAL_AUDIT["mean_js"]<=MAX_JS)
print("Audit",FINAL_AUDIT,"PASS" if AUDIT_PASS else "FAIL")
measures=[]
for batch_size in (1,4):
    examples=SELECT_ITEMS[:batch_size]
    shortest=max(len(e["ids"]) for e in examples)
    for length in (128,256,512):
        if shortest>length:continue
        batch=to_batches(examples,size=batch_size,length=length)[0]
        dense=native.cuda_events_ms(lambda:logits(teacher,batch),
                                    repeats=BENCH_REPEATS)
        fast=native.cuda_events_ms(lambda:logits(student,batch),
                                   repeats=BENCH_REPEATS)
        measures.append(dict(batch=batch_size,tokens=length,
            fp16_ms=dense["median_ms"],hybrid_int4_ms=fast["median_ms"],
            speedup=dense["median_ms"]/fast["median_ms"]))
bench=pd.DataFrame(measures)
assert not bench.empty
bench.to_csv(OUT/"model_speed.csv",index=False)
display(bench)
target=bench[(bench.tokens==512)&bench.batch.isin([1,4])]
SPEED_PASS=(len(target)==2 and bool((target.speedup>=MIN_MODEL_GAIN).all()))
VERIFIED=bool(accepted) and AUDIT_PASS and SPEED_PASS
print("NATIVE INT4 VERIFIED:",VERIFIED,
      "candidate modules:",len(accepted),"audit:",AUDIT_PASS,
      "end-to-end both B1/B4 T512 speed:",SPEED_PASS)


In [ ]:
#@title 8. Diagnostics + executable conversion manifest (source revision pinned)
import torchao
report=dict(starting_repo=STARTING_REPO,revision=SOURCE_SHA,
   gpu=GPU,sm=list(SM),torch=torch.__version__,
   torchao=getattr(torchao,"__version__","unknown"),
   backend_candidates=list(CONFIGS),
   native_int4_selected=accepted,
   selected_count=len(accepted),
   group_size=GROUP_SIZE,real_packed_int4=bool(accepted),
   select=FINAL_DEV,guard=FINAL_GUARD,audit=FINAL_AUDIT,
   audit_pass=bool(AUDIT_PASS),model_speed_pass=bool(SPEED_PASS),
   independently_verified=bool(VERIFIED),
   bench=bench.to_dict(orient="records"),
   limitations=[
     "TorchAO INT4 weights are a new approximation of the starting FP16 model",
     "Native INT4 kernels are hardware- and shape-dependent",
     "M=1 TinyGEMM results do not imply encoder M=512/2048 acceleration",
     "No original Hugging Face model overwritten",
     "No standalone TorchAO int4 checkpoint was published"])
(OUT/"summary_v6.json").write_text(json.dumps(report,indent=2))
(OUT/"conversion_manifest.json").write_text(json.dumps({
  "repo":STARTING_REPO,"revision":SOURCE_SHA,
  "group_size":GROUP_SIZE,
  "torchao_version":report["torchao"],"selected_modules":accepted,
  "validated":bool(VERIFIED)},indent=2))
zip_path=OUT/"Laya_Native_INT4_V6_Results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file() and f!=zip_path:z.write(f,f.name)
print("Results:",zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:pass


## How to read V6

**Fast and accurate:** only if the sealed audit passes **and** both B1×T512 and B4×T512 end-to-end speeds exceed FP16 by the configured threshold. `selected_count` is how many packed CUDA INT4 operators are actually installed.

**No speedup or no supported kernel:** valid research result. The notebook retains FP16 when INT4 would slow the encoder. Increase actual compute capability by switching to L4 / A100 / H100; GemLite is optional and only used if its installed TorchAO API supports it.

**V5 accuracy:** The starting checkpoint defaults to the original Laya FP16, **not** to previously quantized V5 weights. To benchmark native INT4 acceleration of V5 specifically, point `STARTING_REPO` to a complete HF standalone FP16 V5 export.

**Publication:** no automatically published model. TorchAO packed tensor subclasses have version-dependent serialization and require a dedicated, independently tested loader.
